# Imbalance experiments and XGBoost fraud diagnosis
Use Run all and the supplied training files. These experiments keep the previous split dates, duplicate-group folds and selected AutoML hyperparameters. Later-period fraud scores are development checks. The final tables compare threshold-dependent F1 with average precision from raw scores.

**Measured results.** For fraud, the unweighted category-share forest achieved 0.9749 full-training fraud F1. On matched 60,000-row training subsets its F1 fell from 0.9729 to 0.9548 with SMOTE–Tomek. Full-training XGBoost improved from 0.6820 to 0.9727 after adding category history and increasing the leaf limit from 4 to 16. For activity recognition, SMOTE–Tomek led on accuracy (79.85%); balanced training weights led on macro-F1 (0.6826). All figures are development evaluations.


In [1]:
from pathlib import Path
from collections import deque
import zipfile
import shutil
import joblib
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, ExtraTreesClassifier, RandomForestClassifier
from sklearn.feature_selection import VarianceThreshold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

SEED = 610
import os
WORKERS = min(8, os.cpu_count() or 1)
print("scikit-learn", sklearn.__version__)
def locate(name):
    for root in (Path.cwd(), Path.cwd().parent):
        direct = root / name
        if direct.is_file():
            return direct
        data = root / "Docs and Data"
        if data.is_dir():
            match = next(data.rglob(name), None)
            if match:
                return match

if not all(locate(n) for n in ("cct_train.zip", "har_train.csv")):
    from google.colab import files
    files.upload()
fraud_path, har_path = map(locate, ("cct_train.zip", "har_train.csv"))
if not fraud_path or not har_path:
    raise FileNotFoundError("Upload both cct_train.zip and har_train.csv using their original names.")
entity_fields = ["cc_num", "acct_num", "first", "last", "dob", "merchant", "category"]
event_fields = entity_fields + ["unix_time", "trans_time", "trans_date", "amt", "lat", "long", "merch_lat", "merch_long"]
with zipfile.ZipFile(fraud_path) as archive:
    member, = [n for n in archive.namelist() if Path(n).name == "cct_train.csv"]
    with archive.open(member) as stream:
        fraud = pd.read_csv(stream, usecols=event_fields + ["is_fraud"], dtype={k:"string" for k in entity_fields})
fraud = fraud.sort_values("unix_time", kind="stable").reset_index(drop=True)
har = pd.read_csv(har_path, header=None)
print("Fraud rows:", len(fraud), "fraud rate:", fraud.is_fraud.mean())
print("Activity rows:", len(har), "predictors:", har.shape[1]-1)

scikit-learn 1.9.1
Fraud rows: 700807 fraud rate: 0.005106969536548579
Activity rows: 2442 predictors: 328


In [2]:
import importlib.util
import subprocess
import sys
if any(importlib.util.find_spec(k) is None for k in ("imblearn","flaml","xgboost","lightgbm")):
    subprocess.check_call([sys.executable,"-m","pip","install","imbalanced-learn==0.14.2",
                           "flaml==2.8.0","xgboost","lightgbm"])
from imblearn.combine import SMOTETomek
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import TomekLinks
from sklearn.metrics import average_precision_score
from sklearn.model_selection import train_test_split
from flaml.automl.model import XGBoostSklearnEstimator, LGBMEstimator
from collections import Counter

def booster(family, config, task):
    wrapper = {"xgboost":XGBoostSklearnEstimator, "lgbm":LGBMEstimator}[family](
        task=task,n_jobs=WORKERS,**config)
    return wrapper.estimator_class(**wrapper.params)

def resample(matrix, y, strategy, k):
    sampler = SMOTETomek(smote=SMOTE(sampling_strategy=strategy,k_neighbors=k,random_state=SEED),
                         tomek=TomekLinks(sampling_strategy="all",n_jobs=1),random_state=SEED)
    return sampler.fit_resample(matrix,y)

activity_configs=[('xgboost', {'n_estimators': 56, 'max_leaves': 4, 'min_child_weight': 2.707170950755067, 'learning_rate': 0.4810290992611119, 'subsample': 0.9659488304727971, 'colsample_bylevel': 1.0, 'colsample_bytree': 0.9692046632582698, 'reg_alpha': 0.0009765625, 'reg_lambda': 0.1384884162470523}), ('lgbm', {'n_estimators': 100, 'num_leaves': 4, 'min_child_samples': 16, 'learning_rate': 0.07832503615697724, 'log_max_bin': 9, 'colsample_bytree': 0.8814460015050225, 'reg_alpha': 0.0009765625, 'reg_lambda': 0.024037654653852953}), ('xgboost', {'n_estimators': 56, 'max_leaves': 4, 'min_child_weight': 2.707170950755067, 'learning_rate': 0.4810290992611119, 'subsample': 0.9659488304727971, 'colsample_bylevel': 1.0, 'colsample_bytree': 0.9692046632582698, 'reg_alpha': 0.0009765625, 'reg_lambda': 0.1384884162470523}), ('xgboost', {'n_estimators': 56, 'max_leaves': 4, 'min_child_weight': 2.707170950755067, 'learning_rate': 0.4810290992611119, 'subsample': 0.9659488304727971, 'colsample_bylevel': 1.0, 'colsample_bytree': 0.9692046632582698, 'reg_alpha': 0.0009765625, 'reg_lambda': 0.1384884162470523})]
fraud_config={'n_estimators': 96, 'max_leaves': 4, 'min_child_weight': 1.6084394687604544, 'learning_rate': 0.1341120049760053, 'subsample': 0.9632086833875929, 'colsample_bylevel': 1.0, 'colsample_bytree': 1.0, 'reg_alpha': 0.00417650411810458, 'reg_lambda': 0.6036774229389704}

## Activity recognition
Compare the original selected configuration for each grouped outer fold, a scaled control, and SMOTE–Tomek fitted only to that fold’s training rows. SMOTE raises classes below the median training-class count to that count before Tomek cleaning.

In [4]:
Xh, yh = har.iloc[:,:-1], har.iloc[:,-1].to_numpy()
groups = pd.util.hash_pandas_object(Xh, index=False).to_numpy()
labels = np.unique(yh)
# Accept a split only if all activities occur on both sides of every fold.
for split_seed in range(SEED, SEED+200):
    splitter = StratifiedGroupKFold(4, shuffle=True, random_state=split_seed)
    har_folds = list(splitter.split(Xh, yh, groups))
    if all(len(np.unique(yh[tr])) == len(labels) == len(np.unique(yh[va])) for tr,va in har_folds):
        break
else:
    raise ValueError("No four-fold split covers all activities.")
print("Fold seed:", split_seed)
labels = np.unique(yh)
ycode = np.searchsorted(labels,yh)
preds = {k:np.empty_like(ycode) for k in ("Original","Scaled control","SMOTE-Tomek")}
for fold,(tr,va) in enumerate(har_folds):
    assert not np.intersect1d(groups[tr],groups[va]).size
    fam,cfg = activity_configs[fold]
    x, v = Xh.iloc[tr].to_numpy(),Xh.iloc[va].to_numpy()
    scale=StandardScaler().fit(x)
    xs,vs=scale.transform(x),scale.transform(v)
    counts=Counter(ycode[tr]); target=int(np.median(list(counts.values())))
    strategy={c:target for c,n in counts.items() if n<target}
    xr,yr=resample(xs,ycode[tr],strategy,min(3,min(counts.values())-1))
    for name,train,y,valid in [("Original",x,ycode[tr],v),("Scaled control",xs,ycode[tr],vs),
                               ("SMOTE-Tomek",xr,yr,vs)]:
        preds[name][va]=booster(fam,cfg,"multiclass").fit(train,y).predict(valid)
display(pd.DataFrame([dict(setting=k,correct=int((v==ycode).sum()),accuracy=accuracy_score(ycode,v),
                           macro_f1=f1_score(ycode,v,average="macro")) for k,v in preds.items()]))

Fold seed: 610
       setting  correct  accuracy  macro_f1
      Original     1919  0.785831  0.645098
Scaled control     1926  0.788698  0.654241
   SMOTE-Tomek     1950  0.798526  0.668414


## Fraud detection
Run full-training XGBoost with and without recent category frequency and a larger leaf limit. Compare against the category-share forest. Separately test moderate SMOTE–Tomek on identical stratified 60,000-row training subsets for both models. Validation and holdout rows retain their original class prevalence. Synthetic vectors are created after chronological histories are calculated and never become events in those histories. Tune each fraud threshold on pooled March–August predictions.

In [6]:
def fraud_features(events):
    # Return features in input order, including when called on a shuffled batch.
    d = events.reset_index(drop=True).sort_values("unix_time", kind="stable")
    x = pd.DataFrame(index=d.index)
    x["amount"] = d.amt
    x["hour"] = d.trans_time.str.slice(0, 2).astype(int)
    lat1, lat2 = np.radians(d.lat), np.radians(d.merch_lat)
    a = np.sin((lat2-lat1)/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(np.radians(d.merch_long-d.long)/2)**2
    x["home_merchant_km"] = 12742 * np.arcsin(np.sqrt(a.clip(0, 1)))

    d["time_band"] = x["hour"] // 6
    entities = {"card":["cc_num"], "card_category":["cc_num","category"],
                "account":["acct_num"], "person":["first","last","dob"],
                "card_merchant":["cc_num","merchant"], "category":["category"], "merchant":["merchant"],
                "merchant_time":["merchant","time_band"], "card_time":["cc_num","time_band"]}
    for prefix, keys in entities.items():
        # Collapse simultaneous events so their within-second ordering is immaterial.
        group_keys = keys + ["unix_time"]
        present = d[keys].notna().all(axis=1)
        batch = (d.loc[present].groupby(group_keys, sort=False, observed=True)
                 .amt.agg(total="sum", n="size").reset_index())
        groups = batch.groupby(keys, sort=False, observed=True)
        count = groups.n.cumsum() - batch.n
        total = groups.total.cumsum() - batch.total
        batch["count"] = count
        batch["mean"] = total / count.replace(0, np.nan)
        batch["elapsed"] = groups.unix_time.diff()
        prior = d[group_keys].merge(batch[group_keys + ["count","mean","elapsed"]],
                                    on=group_keys, how="left", sort=False, validate="many_to_one")
        prior.index = d.index
        x[prefix+"_count"] = prior["count"].fillna(0)
        x[prefix+"_amount_ratio"] = d.amt / prior["mean"].replace(0, np.nan)
        if prefix == "card":
            x["card_elapsed"] = prior["elapsed"]
    # Prefix sums and binary search make strict trailing windows inexpensive per card.
    for name in ("hour_count","day_count","day_spend"):
        x[name] = 0.
    x["reference_days_observed"] = 0.
    x["day_vs_prior28_ratio"] = np.nan
    for _, block in d.groupby("cc_num", sort=False):
        times = block.unix_time.to_numpy()
        end = times.searchsorted(times, side="left")
        hour = times.searchsorted(times-3600)
        day = times.searchsorted(times-86400)
        sums = np.r_[0., block.amt.to_numpy().cumsum()]
        x.loc[block.index, ["hour_count","day_count","day_spend"]] = np.column_stack(
            [end-hour, end-day, sums[end]-sums[day]])
        ref_start = times.searchsorted(times-29*86400, side="left")
        ref_total = sums[day]-sums[ref_start]
        coverage = np.clip((times-times[0])/86400-1, 0, 28)
        ratio = np.full(len(block), np.nan)
        np.divide(sums[end]-sums[day], ref_total/28, out=ratio,
                  where=(coverage == 28) & (ref_total > 0))
        x.loc[block.index, ["reference_days_observed","day_vs_prior28_ratio"]] = np.column_stack([coverage,ratio])
    return x.sort_index().astype("float32")

for keys in [["first","last"], ["first","last","dob"], ["acct_num"]]:
    card_counts = fraud.groupby(keys).cc_num.nunique()
    print(keys, "groups:", len(card_counts), "groups with multiple cards:", (card_counts > 1).sum())
Xf = fraud_features(fraud[event_fields])
yf = fraud.is_fraud.to_numpy(dtype=int)
basic = ["amount","hour"]
card = basic + ["card_amount_ratio","card_category_amount_ratio","hour_count","day_count","card_elapsed"]
extended = card + ["day_spend","card_count","card_merchant_count","home_merchant_km",
                   "account_count","account_amount_ratio","person_count","person_amount_ratio",
                   "category_amount_ratio"]
print("Feature sets:", len(basic), len(card), len(extended))

['first', 'last'] groups: 496 groups with multiple cards: 3
['first', 'last', 'dob'] groups: 500 groups with multiple cards: 0
['acct_num'] groups: 500 groups with multiple cards: 0
Feature sets: 2 7 16


In [7]:
def best_threshold(y, score, objective="accuracy"):
    order = np.argsort(-score, kind="stable")
    ranked, truth = score[order], y[order]
    ends = np.flatnonzero(np.r_[ranked[:-1] != ranked[1:], True])
    tp = np.r_[0, np.cumsum(truth)[ends]]
    positives = np.r_[0, ends+1]
    correct = (len(y)-y.sum()) + 2*tp - positives
    tied = np.flatnonzero(correct == correct.max())
    f1 = 2*tp / np.maximum(1, positives+y.sum())
    if objective == "fraud_f1":
        tied = np.flatnonzero(f1 == f1.max())
        winner = tied[np.argmax(correct[tied])]
    else:
        winner = tied[np.argmax(f1[tied])]
    return np.r_[np.inf, ranked[ends]][winner]

def fraud_metrics(y, score, threshold):
    pred = score >= threshold
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0,1]).ravel()
    return dict(correct=int(tn+tp), accuracy=accuracy_score(y,pred), fraud_f1=f1_score(y,pred,zero_division=0),
                macro_f1=f1_score(y,pred,average="macro"), auc=roc_auc_score(y,score),
                false_alerts=fp, missed_frauds=fn, caught=tp)

dates = pd.to_datetime(fraud.trans_date)
edges = pd.to_datetime(["2022-03-01","2022-05-01","2022-07-01","2022-09-01"])
folds = [(np.flatnonzero(dates < a), np.flatnonzero((dates >= a) & (dates < b)))
         for a,b in zip(edges[:-1], edges[1:])]
def fit_fraud(name, rows):
    model, cols = fraud_models[name]
    return clone(model).fit(Xf.iloc[rows][cols], yf[rows]) if model is not None else None

def fraud_scores(model, frame, cols):
    if model is None:
        return frame["amount"].to_numpy()
    return model.predict_proba(frame[cols])[:,1]

from sklearn.impute import SimpleImputer
from itertools import product

def forest(leaf):
    return make_pipeline(SimpleImputer(strategy="median", add_indicator=True),
                         RandomForestClassifier(n_estimators=100, max_samples=.5,
                                                min_samples_leaf=leaf, n_jobs=WORKERS, random_state=SEED))

base_forest, merchant_forest, trend_forest = "Original", "Merchant history", "Card spending trend"
merchant = ["merchant_count", "merchant_amount_ratio"]
trend = ["reference_days_observed","day_vs_prior28_ratio"]
combined = extended + merchant + trend
context = combined + ["card_merchant_amount_ratio","card_category_count",
                      "merchant_time_count","merchant_time_amount_ratio","card_time_amount_ratio"]
fraud_models = {base_forest:(forest(4),extended), merchant_forest:(forest(4),extended+merchant),
                trend_forest:(forest(4),extended+trend)}
for feature_name, columns in [("Combined", combined), ("Combined + context", context)]:
    for leaf in (4,2,8):
        fraud_models[f"{feature_name}, leaf {leaf}"] = (forest(leaf),columns)

In [8]:
# Category frequency features use only timestamps strictly earlier than the scored row.
t = fraud.unix_time.to_numpy()
def prior_counts(times,days):
    return times.searchsorted(times)-times.searchsorted(times-days*86400)
c7=np.zeros(len(t),int); c35=c7.copy()
for idx in fraud.groupby("category",sort=False).indices.values():
    c7[idx]=prior_counts(t[idx],7); c35[idx]=prior_counts(t[idx],35)
n7,n35=prior_counts(t,7),prior_counts(t,35)
Xf["recent_category_share"]=(c7+.5)/(n7+7)
Xf["category_share_change"]=np.log(Xf.recent_category_share/((c35-c7+.5)/(n35-n7+7)))
newcols=context+["recent_category_share","category_share_change"]
old=Xf[context].fillna(-1).to_numpy(dtype="float32")
new=Xf[newcols].fillna(-1).to_numpy(dtype="float32")
# Use the original forest's median imputer and missingness flags for its full-data benchmark.
new_raw=Xf[newcols].to_numpy(dtype="float32")
all_splits=folds+[(np.flatnonzero(dates<edges[-1]),np.flatnonzero(dates>=edges[-1]))]
scores={}
def keep(name,model,x,y,v):
    scores.setdefault(name,[]).append(model.fit(x,y).predict_proba(v)[:,1])

for fold,(tr,va) in enumerate(all_splits):
    assert dates.iloc[tr].max() < dates.iloc[va].min()
    keep("XGB original",booster("xgboost",fraud_config,"binary"),old[tr],yf[tr],old[va])
    keep("XGB + category",booster("xgboost",fraud_config,"binary"),new[tr],yf[tr],new[va])
    keep("XGB 16 leaves original features",booster("xgboost",{**fraud_config,"max_leaves":16},"binary"),old[tr],yf[tr],old[va])
    keep("XGB 16 leaves + category",booster("xgboost",{**fraud_config,"max_leaves":16},"binary"),new[tr],yf[tr],new[va])
    keep("RF + category",forest(2),new_raw[tr],yf[tr],new_raw[va])
    sub,_=train_test_split(tr,train_size=60000,stratify=yf[tr],random_state=SEED)
    sub=np.sort(sub)
    # Impute and standardize only on the sampled training rows for distance-based resampling.
    prep=make_pipeline(SimpleImputer(strategy="median",add_indicator=True),StandardScaler())
    xs=prep.fit_transform(new_raw[sub]);vs=prep.transform(new_raw[va])
    xr,yr=resample(xs,yf[sub],0.10,5)
    print("Fold",fold+1,"training counts",dict(Counter(yf[sub])),"after resampling",dict(Counter(yr)),flush=True)
    for label,x,y in [("control",xs,yf[sub]),("SMOTE-Tomek",xr,yr)]:
        keep("60k RF "+label,forest(2),x,y,vs)
        keep("60k XGB "+label,booster("xgboost",fraud_config,"binary"),x,y,vs)
truth=np.concatenate([yf[va] for _,va in folds])
test=yf[all_splits[-1][1]]
rows=[]
for name,s in scores.items():
    pooled=np.concatenate(s[:-1]); cut=best_threshold(truth,pooled,"fraud_f1")
    met=fraud_metrics(test,s[-1],cut)
    rows.append(dict(model=name,threshold=cut,dev_F1=f1_score(truth,pooled>=cut),dev_AP=average_precision_score(truth,pooled),
                     **met, AP=average_precision_score(test,s[-1]),F1_at_half=f1_score(test,s[-1]>=.5)))
display(pd.DataFrame(rows))

Fold 1 training counts {np.int64(0): 59660, np.int64(1): 340} after resampling {np.int64(0): 59660, np.int64(1): 5966}
Fold 2 training counts {np.int64(0): 59665, np.int64(1): 335} after resampling {np.int64(0): 59664, np.int64(1): 5965}
Fold 3 training counts {np.int64(0): 59668, np.int64(1): 332} after resampling {np.int64(0): 59667, np.int64(1): 5965}
Fold 4 training counts {np.int64(0): 59671, np.int64(1): 329} after resampling {np.int64(0): 59669, np.int64(1): 5965}
                          model  threshold   dev_F1   dev_AP  correct  accuracy  fraud_f1  macro_f1      auc  false_alerts  missed_frauds  caught       AP  F1_at_half
                   XGB original   0.366673 0.817924 0.856733   140092  0.997536  0.681985  0.840374 0.993534           210            136     371 0.674528    0.657769
                 XGB + category   0.836713 0.955714 0.966854   140404  0.999758  0.965863  0.982871 0.998303             8             26     481 0.976754    0.960239
XGB 16 leaves original 

## Class weights on the same training rows
Fraud uses the same 60,000-row training subsets, preprocessing and evaluation rows as the sampler. Compare inverse-frequency weights and square-root weights. Activity uses the same grouped folds and scaled features.

In [9]:
from sklearn.utils.class_weight import compute_sample_weight
weight_scores={}
for tr,va in all_splits:
    sub,_=train_test_split(tr,train_size=60000,stratify=yf[tr],random_state=SEED)
    sub=np.sort(sub)
    prep=make_pipeline(SimpleImputer(strategy="median",add_indicator=True),StandardScaler())
    xs,vs=prep.fit_transform(new_raw[sub]),prep.transform(new_raw[va])
    y=yf[sub]; ratio=(y==0).sum()/(y==1).sum()
    for policy,mult in [("sqrt",np.sqrt(ratio)),("balanced",ratio)]:
        rf=forest(2).set_params(randomforestclassifier__class_weight={0:1.,1:mult})
        xgb=booster("xgboost",{**fraud_config,"scale_pos_weight":mult},"binary")
        for family,model in [("RF",rf),("XGB",xgb)]:
            weight_scores.setdefault(family+" "+policy,[]).append(model.fit(xs,y).predict_proba(vs)[:,1])
wr=[]
for name,s in weight_scores.items():
    cal=np.concatenate(s[:-1]);cut=best_threshold(truth,cal,"fraud_f1");met=fraud_metrics(test,s[-1],cut)
    wr.append(dict(model=name,cut=cut,dev_F1=f1_score(truth,cal>=cut),**met,
                   precision=met["caught"]/(met["caught"]+met["false_alerts"]),AP=average_precision_score(test,s[-1])))
display(pd.DataFrame(wr))

# Secondary activity check with exactly the same groups and scaled control preprocessing.
hw={key:np.empty_like(ycode) for key in ("sqrt","balanced")}
for fold,(tr,va) in enumerate(har_folds):
    fam,cfg=activity_configs[fold]
    scale=StandardScaler().fit(Xh.iloc[tr].to_numpy())
    x,v=scale.transform(Xh.iloc[tr].to_numpy()),scale.transform(Xh.iloc[va].to_numpy())
    balance=compute_sample_weight("balanced",ycode[tr])
    for policy,w in [("sqrt",np.sqrt(balance)),("balanced",balance)]:
        hw[policy][va]=booster(fam,cfg,"multiclass").fit(x,ycode[tr],sample_weight=w/w.mean()).predict(v)
display(pd.DataFrame([dict(setting=k,correct=int((v==ycode).sum()),accuracy=accuracy_score(ycode,v),
                           macro_f1=f1_score(ycode,v,average="macro")) for k,v in hw.items()]))

       model      cut   dev_F1  correct  accuracy  fraud_f1  macro_f1      auc  false_alerts  missed_frauds  caught  precision       AP
     RF sqrt 0.571674 0.957089   140398  0.999715  0.960396  0.980127 0.993448            18             22     485   0.964215 0.968114
    XGB sqrt 0.833500 0.956050   140403  0.999751  0.964895  0.982385 0.998919             9             26     481   0.981633 0.974481
 RF balanced 0.468109 0.938082   140377  0.999566  0.939303  0.969543 0.993444            26             35     472   0.947791 0.964381
XGB balanced 0.967951 0.947888   140399  0.999722  0.960725  0.980293 0.998929             9             30     477   0.981481 0.972939
 setting  correct  accuracy  macro_f1
    sqrt     1922  0.787060  0.671461
balanced     1920  0.786241  0.682605
